# 2.3 — File and CSV input/output

Values used so far disappeared when a program ended. In this lesson, read book records from an external file, validate them, save a separate CSV, and prove that the result can be loaded again. Never overwrite the supplied source data.

## Introduction

Use this Notebook to verify the lesson ideas with executable code.

## Learning outcomes

- Resolve file paths from a defined base rather than an assumed current directory.
- Open and close text files safely with an explicit mode and UTF-8 encoding.
- Read CSV with DictReader and convert field values according to a declared schema.
- Validate the header and every row before using the records.
- Write a separate CSV with DictWriter and verify the saved product by re-reading it.

> **Learning route:** Required: 2.3.1–2.3.5  |  Integration: 2.3.6


## 2.3.1 Resolve the intended file path

A relative path is interpreted from the current working directory. An English Notebook and one inside the `ja` folder may start in different places, so search upward for the supplied course file. When a path fails, display the resolved absolute path before guessing a replacement.

In [ ]:
from pathlib import Path

def find_course_file(relative_path):
    """Find a supplied course file from a Notebook opened at any course level."""
    start = Path.cwd().resolve()
    for folder in [start, *start.parents]:
        candidate = folder / relative_path
        if candidate.is_file():
            return candidate
    raise FileNotFoundError(f"Course file not found: {relative_path}; started at {start}")

source_path = find_course_file(Path("data") / "library-books-practice.csv")
print("Resolved source:", source_path)


### In a `.py` file, resolve paths from `__file__`

The Notebook searched from `Path.cwd()`. An independent script should resolve paths from its own location. The Project 2.4 starter supplies this pattern.

```python
BASE_DIR = Path(__file__).resolve().parent
INPUT_PATH = BASE_DIR / "data" / "books.csv"
OUTPUT_PATH = BASE_DIR / "output" / "books_updated.csv"
```

## 2.3.2 Open text files with an explicit mode and encoding

Leaving a `with` block closes the file automatically. Mode `r` reads, `w` creates or replaces, and `a` appends. State UTF-8 explicitly. For CSV, pass `newline=""` so the CSV module controls record endings.

In [ ]:
with source_path.open("r", encoding="utf-8", newline="") as file:
    text = file.read()

print(text)


## 2.3.3 Read CSV records without losing their structure

CSV can quote a field that contains a comma. The title in the second sample row contains a comma but remains one title. The standard `csv` module handles that rule correctly.

In [ ]:
import csv

with source_path.open("r", encoding="utf-8", newline="") as file:
    reader = csv.DictReader(file)
    print("Header:", reader.fieldnames)
    raw_rows = list(reader)

for row in raw_rows:
    print(row)


### Values read from CSV begin as strings

`DictReader` uses the header as dictionary keys, but it does not turn `false` into `False`. `bool("false")` is `True` because the string is non-empty. Write a conversion function that checks meaning and raises `ValueError` for unsupported text.

In [ ]:
def parse_read(value):
    normalised = value.strip().lower()
    if normalised == "true":
        return True
    if normalised == "false":
        return False
    raise ValueError(f"read must be true or false: {value!r}")

print(parse_read(" TRUE "))
print(parse_read("false"))

try:
    parse_read("yes")
except ValueError as error:
    print(type(error).__name__, error)


## 2.3.4 Validate headers, rows, and converted values

The required columns are `id`, `title`, and `read`. Silently repairing a missing column, blank ID or title, duplicate ID, or invalid Boolean lets later work continue with false data. Reject it at the input boundary with a useful cause. Extra columns may be ignored here.

In [ ]:
REQUIRED_FIELDS = {"id", "title", "read"}

def validate_header(fieldnames):
    actual = set(fieldnames or [])
    missing = REQUIRED_FIELDS - actual
    if missing:
        raise ValueError(f"Missing CSV columns: {sorted(missing)}")

def load_books(path):
    books = []
    seen_ids = set()
    with path.open("r", encoding="utf-8", newline="") as file:
        reader = csv.DictReader(file)
        validate_header(reader.fieldnames)
        for line_number, row in enumerate(reader, start=2):
            book_id = row["id"].strip()
            title = row["title"].strip()
            if not book_id or not title:
                raise ValueError(f"Blank required value on line {line_number}")
            if book_id in seen_ids:
                raise ValueError(f"Duplicate id on line {line_number}: {book_id}")
            books.append({"id": book_id, "title": title, "read": parse_read(row["read"])})
            seen_ids.add(book_id)
    return books

books = load_books(source_path)
print(books)


### Test a missing header independently

There is no need to damage the supplied file. Pass a test header to the validation function and confirm that the expected exception is raised.

In [ ]:
try:
    validate_header(["id", "title"])
except ValueError as error:
    print(type(error).__name__, error)


## 2.3.5 Save to a separate file and verify by re-reading

The supplied CSV in `data` is input evidence. Save changed records under `output`. Give `DictWriter` a stable field order and convert Python Booleans back to lower-case `true` or `false`.

In [ ]:
def save_books(books, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8", newline="") as file:
        writer = csv.DictWriter(file, fieldnames=["id", "title", "read"])
        writer.writeheader()
        for book in books:
            writer.writerow({
                "id": book["id"],
                "title": book["title"],
                "read": "true" if book["read"] else "false",
            })

source_before = source_path.read_bytes()
updated_books = []
for book in books:
    updated_books.append(book.copy())
updated_books[2]["read"] = True

output_path = source_path.parents[1] / "output" / "lesson23-books-updated.csv"
save_books(updated_books, output_path)
print("Saved:", output_path)


### Reload and compare instead of trusting a successful save

A file can exist while containing the wrong columns or text conversion. Reload it with the same `load_books()` function and compare it with the expected records. Also prove that the supplied CSV bytes are unchanged.

In [ ]:
reloaded_books = load_books(output_path)
assert reloaded_books == updated_books
assert source_path.read_bytes() == source_before
assert reloaded_books[2]["read"] is True
print("ROUND TRIP OK")
print("SOURCE PRESERVED")


### For `FileNotFoundError`, display where Python looked

A filename alone does not reveal the directory Python used. Resolve and display the candidate, then check distribution, spelling, and letter case in that order.

In [ ]:
missing_path = source_path.parent / "missing.csv"
print("Would read:", missing_path.resolve())
print("Exists:", missing_path.exists())


## 2.3.6 Integrated practice: build a complete CSV round trip

Load `data/library-books-practice.csv`, change only the title of `L001` to `Python Foundations` in a copied record collection, and save `output/lesson23-practice.csv`. Do not change the source CSV. Reload the output with `load_books()` and use `assert` to check record count, ID order, title, and Boolean values.

In [ ]:
# Write the transfer solution here.


## Summary

- Separated path resolution, opening, CSV parsing, conversion, and validation.
- Preserved quoted CSV fields that contain commas.
- Protected the source file and validated the output after a complete round trip.

## Next

Project 2.4 combines record structures, tested functions, validation, and CSV input/output in a library-record update program.

**Estimated learning time:** about 3 hours
